# Healthcare Analytics

This notebook provides analytical insights using the Gold-layer
Star Schema.

The analytics focus on key healthcare business areas:

1. Patient visit analysis
2. Doctor performance
3. Department performance
4. Treatment analysis
5. Medication usage
6. Laboratory testing
7. Revenue and billing
8. Insurance claims
9. Disease and diagnosis trends
10. Healthcare KPIs

All analytical queries use the Gold fact and dimension tables rather
than directly querying the normalized OLTP source.

In [0]:
from pyspark.sql import functions as F

fact_visits = spark.table("fact_patient_visits")

visit_kpis = fact_visits.agg(
    F.countDistinct("appointment_id").alias("total_visits"),
    F.sum("is_completed").alias("completed_visits"),
    F.sum("is_cancelled").alias("cancelled_visits"),
    F.round(
        F.avg("wait_time_minutes"), 2
    ).alias("average_wait_time_minutes"),
    F.round(
        F.avg("consultation_duration_minutes"), 2
    ).alias("average_consultation_duration_minutes")
)

visit_kpis = visit_kpis.withColumn(
    "completion_rate_percentage",
    F.round(
        F.col("completed_visits")
        / F.col("total_visits") * 100,
        2
    )
)

display(visit_kpis)

total_visits,completed_visits,cancelled_visits,average_wait_time_minutes,average_consultation_duration_minutes,completion_rate_percentage
500,345,45,45.84,32.76,69.0


## KPI 2 — Doctor Performance

Doctor performance is analyzed using patient visit activity.

The analysis measures:

- Total visits handled
- Completed visits
- Cancelled visits
- Average waiting time
- Average consultation duration
- Visit completion rate

These metrics can help hospital management evaluate workload and
operational performance across doctors.

In [0]:
doctor_performance = (
    fact_visits.alias("f")
    .join(
        spark.table("dim_doctor").alias("d"),
        F.col("f.doctor_id") == F.col("d.doctor_id"),
        "left"
    )
    .groupBy(
        F.col("f.doctor_id").alias("doctor_id"),
        F.concat(
            F.col("d.first_name"),
            F.lit(" "),
            F.col("d.last_name")
        ).alias("doctor_name"),
        F.col("d.specialization").alias("specialization")
    )
    .agg(
        F.countDistinct(
            F.col("f.appointment_id")
        ).alias("total_visits"),

        F.sum(
            F.col("f.is_completed")
        ).alias("completed_visits"),

        F.sum(
            F.col("f.is_cancelled")
        ).alias("cancelled_visits"),

        F.round(
            F.avg(F.col("f.wait_time_minutes")),
            2
        ).alias("average_wait_time_minutes"),

        F.round(
            F.avg(F.col("f.consultation_duration_minutes")),
            2
        ).alias("average_consultation_duration_minutes")
    )
    .withColumn(
        "completion_rate_percentage",
        F.round(
            F.col("completed_visits")
            / F.col("total_visits") * 100,
            2
        )
    )
    .orderBy(F.desc("total_visits"))
)

display(doctor_performance)

doctor_id,doctor_name,specialization,total_visits,completed_visits,cancelled_visits,average_wait_time_minutes,average_consultation_duration_minutes,completion_rate_percentage
1,Ananya Sharma,Neurology,500,345,45,45.84,32.76,69.0
1,Ananya Sharma,General Medicine,500,345,45,45.84,32.76,69.0


## KPI 3 — Department Performance

Department performance is analyzed using patient visit activity.

The analysis measures:

- Total visits
- Completed visits
- Cancelled visits
- Average waiting time
- Average consultation duration
- Visit completion rate

These metrics help hospital management understand department
workload and operational efficiency.

In [0]:
department_performance = (
    fact_visits.alias("f")
    .join(
        spark.table("dim_department").alias("dept"),
        F.col("f.department_id") == F.col("dept.department_id"),
        "left"
    )
    .groupBy(
        F.col("f.department_id").alias("department_id"),
        F.col("dept.department_name").alias("department_name")
    )
    .agg(
        F.countDistinct(
            F.col("f.appointment_id")
        ).alias("total_visits"),

        F.sum(
            F.col("f.is_completed")
        ).alias("completed_visits"),

        F.sum(
            F.col("f.is_cancelled")
        ).alias("cancelled_visits"),

        F.round(
            F.avg(F.col("f.wait_time_minutes")),
            2
        ).alias("average_wait_time_minutes"),

        F.round(
            F.avg(F.col("f.consultation_duration_minutes")),
            2
        ).alias("average_consultation_duration_minutes")
    )
    .withColumn(
        "completion_rate_percentage",
        F.round(
            F.col("completed_visits")
            / F.col("total_visits") * 100,
            2
        )
    )
    .orderBy(F.desc("total_visits"))
)

display(department_performance)

department_id,department_name,total_visits,completed_visits,cancelled_visits,average_wait_time_minutes,average_consultation_duration_minutes,completion_rate_percentage
1,General Medicine,500,345,45,45.84,32.76,69.0



## KPI 4 — Treatment Analysis

Treatment activity is analyzed using the treatment fact table.

The analysis measures:

- Total treatment records
- Total quantity performed
- Completed treatments
- Cancelled treatments
- Treatment activity by treatment type

This helps healthcare management understand treatment utilization
and operational demand.

In [0]:
fact_treatments = spark.table("fact_treatments")

treatment_analysis = (
    fact_treatments.alias("f")
    .join(
        spark.table("dim_treatment").alias("t"),
        F.col("f.treatment_id") == F.col("t.treatment_id"),
        "left"
    )
    .groupBy(
        F.col("f.treatment_id").alias("treatment_id"),
        F.col("t.treatment_name").alias("treatment_name"),
        F.col("t.treatment_category").alias("treatment_category")
    )
    .agg(
        F.countDistinct(
            F.col("f.appointment_treatment_id")
        ).alias("total_treatment_records"),

        F.sum(
            F.col("f.quantity")
        ).alias("total_quantity"),

        F.sum(
            F.when(
                F.upper(F.col("f.treatment_status")) == "COMPLETED",
                1
            ).otherwise(0)
        ).alias("completed_treatments"),

        F.sum(
            F.when(
                F.upper(F.col("f.treatment_status")) == "CANCELLED",
                1
            ).otherwise(0)
        ).alias("cancelled_treatments")
    )
    .orderBy(F.desc("total_quantity"))
)

display(treatment_analysis)

## KPI 5 — Medication Usage

Medication usage is analyzed using the prescription fact table.

The analysis measures:

- Number of prescription records
- Total quantity prescribed
- Number of unique prescriptions
- Medication usage by medication type

These metrics help identify medication demand and prescription
patterns.

In [0]:
fact_prescriptions = spark.table("fact_prescriptions")

medication_usage = (
    fact_prescriptions.alias("f")
    .join(
        spark.table("dim_medication").alias("m"),
        F.col("f.medication_id") == F.col("m.medication_id"),
        "left"
    )
    .groupBy(
        F.col("f.medication_id").alias("medication_id"),
        F.col("m.generic_name").alias("generic_name"),
        F.col("m.brand_name").alias("brand_name"),
        F.col("m.dosage_form").alias("dosage_form")
    )
    .agg(
        F.countDistinct(
            F.col("f.prescription_item_id")
        ).alias("prescription_records"),

        F.countDistinct(
            F.col("f.prescription_id")
        ).alias("unique_prescriptions"),

        F.sum(
            F.col("f.quantity")
        ).alias("total_quantity_prescribed")
    )
    .orderBy(F.desc("total_quantity_prescribed"))
)

display(medication_usage)

medication_id,generic_name,brand_name,dosage_form,prescription_records,unique_prescriptions,total_quantity_prescribed
1,Metformin,HealthMed-Metformin,Tablet,600,300,17665


## KPI 6 — Laboratory Testing Analysis

Laboratory activity is analyzed using the laboratory test fact table.

The analysis measures:

- Total laboratory tests
- Completed tests
- Pending tests
- Total laboratory cost
- Average laboratory cost
- Tests by category

These metrics help management understand laboratory workload,
testing demand, and associated costs.

In [0]:
fact_lab_tests = spark.table("fact_lab_tests")

lab_analysis = (
    fact_lab_tests
    .groupBy(
        "test_category"
    )
    .agg(
        F.countDistinct(
            "lab_test_id"
        ).alias("total_tests"),

        F.sum(
            F.when(
                F.upper(F.col("test_status")) == "COMPLETED",
                1
            ).otherwise(0)
        ).alias("completed_tests"),

        F.sum(
            F.when(
                F.upper(F.col("test_status")) == "PENDING",
                1
            ).otherwise(0)
        ).alias("pending_tests"),

        F.round(
            F.sum("lab_cost"),
            2
        ).alias("total_lab_cost"),

        F.round(
            F.avg("lab_cost"),
            2
        ).alias("average_lab_cost")
    )
    .orderBy(F.desc("total_tests"))
)

display(lab_analysis)

test_category,total_tests,completed_tests,pending_tests,total_lab_cost,average_lab_cost
Hematology,144,123,0,399022.02,2770.99
Biochemistry,132,106,0,340253.15,2577.68
Cardiology,124,101,0,318863.68,2571.48


## KPI 7 — Revenue and Billing Analysis

Healthcare billing performance is analyzed using the billing fact table.

The analysis measures:

- Total billed amount
- Total insurance amount
- Total patient amount
- Total discounts
- Total tax
- Number of billing transactions
- Average bill amount

These metrics help management understand hospital revenue,
insurance contribution, patient contribution, and billing trends.

In [0]:
fact_billing = spark.table("fact_billing")

billing_kpis = (
    fact_billing
    .agg(
        F.countDistinct(
            "billing_id"
        ).alias("total_billing_transactions"),

        F.round(
            F.sum("gross_amount"),
            2
        ).alias("total_gross_amount"),

        F.round(
            F.sum("discount_amount"),
            2
        ).alias("total_discount_amount"),

        F.round(
            F.sum("tax_amount"),
            2
        ).alias("total_tax_amount"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_revenue"),

        F.round(
            F.sum("insurance_amount"),
            2
        ).alias("total_insurance_amount"),

        F.round(
            F.sum("patient_amount"),
            2
        ).alias("total_patient_amount"),

        F.round(
            F.avg("total_amount"),
            2
        ).alias("average_bill_amount")
    )
)

display(billing_kpis)

total_billing_transactions,total_gross_amount,total_discount_amount,total_tax_amount,total_revenue,total_insurance_amount,total_patient_amount,average_bill_amount
500,2.501637113E7,1991748.38,2050463.27,2.507508602E7,1.542050001E7,9654586.01,50150.17


### Billing by Payment Status

Billing transactions are grouped by payment status to identify
the distribution of paid, pending, or other billing states.

In [0]:
payment_status_analysis = (
    fact_billing
    .groupBy("payment_status")
    .agg(
        F.countDistinct(
            "billing_id"
        ).alias("billing_transactions"),

        F.round(
            F.sum("total_amount"),
            2
        ).alias("total_amount"),

        F.round(
            F.sum("patient_amount"),
            2
        ).alias("patient_amount")
    )
    .orderBy(F.desc("total_amount"))
)

display(payment_status_analysis)

payment_status,billing_transactions,total_amount,patient_amount
Paid,298,1.495700838E7,5827875.65
Pending,93,4971423.1,1665499.84
Partially Paid,97,4572016.98,1923116.36
Cancelled,12,574637.56,238094.16


## KPI 8 — Insurance Claims Analysis

Insurance claim activity is analyzed using the insurance claims
fact table.

The analysis measures:

- Total claims
- Total claimed amount
- Total approved amount
- Pending or rejected claims
- Claim approval rate
- Average claim amount
- Average approved amount

These metrics help management understand insurance utilization,
claim processing, and reimbursement performance.

In [0]:
fact_claims = spark.table("fact_insurance_claims")

claims_kpis = (
    fact_claims
    .agg(
        F.countDistinct(
            "claim_id"
        ).alias("total_claims"),

        F.round(
            F.sum("claim_amount"),
            2
        ).alias("total_claimed_amount"),

        F.round(
            F.sum("approved_amount"),
            2
        ).alias("total_approved_amount"),

        F.round(
            F.avg("claim_amount"),
            2
        ).alias("average_claim_amount"),

        F.round(
            F.avg("approved_amount"),
            2
        ).alias("average_approved_amount")
    )
    .withColumn(
        "approval_rate_percentage",
        F.round(
            F.col("total_approved_amount")
            / F.col("total_claimed_amount") * 100,
            2
        )
    )
)

display(claims_kpis)

total_claims,total_claimed_amount,total_approved_amount,average_claim_amount,average_approved_amount,approval_rate_percentage
200,7669228.38,6574764.54,38346.14,32873.82,85.73


### Claims by Status

Insurance claims are grouped by their processing status to identify
approved, pending, rejected, or other claim states.

In [0]:
claims_by_status = (
    fact_claims
    .groupBy("claim_status")
    .agg(
        F.countDistinct(
            "claim_id"
        ).alias("total_claims"),

        F.round(
            F.sum("claim_amount"),
            2
        ).alias("total_claimed_amount"),

        F.round(
            F.sum("approved_amount"),
            2
        ).alias("total_approved_amount")
    )
    .orderBy(F.desc("total_claimed_amount"))
)

display(claims_by_status)

claim_status,total_claims,total_claimed_amount,total_approved_amount
Approved,100,3815151.67,3246847.8
Submitted,54,1882867.76,1644314.08
Partially Approved,26,1085262.84,904396.61
Rejected,20,885946.11,779206.05


## KPI 9 — Disease and Diagnosis Trends

Disease trends are analyzed using appointment diagnosis records and
the Diagnosis Dimension.

The analysis measures:

- Number of diagnosis records
- Number of unique appointments
- Number of unique patients
- Disease category
- Chronicity
- Severity

These metrics help identify common diseases and understand disease
patterns across the healthcare population.

In [0]:
appointment_diagnoses = spark.table("silver_appointment_diagnoses")
dim_diagnosis = spark.table("dim_diagnosis")

diagnosis_trends = (
    appointment_diagnoses.alias("ad")
    .join(
        dim_diagnosis.alias("d"),
        F.col("ad.diagnosis_id") == F.col("d.diagnosis_id"),
        "left"
    )
    .join(
        spark.table("silver_appointments").select(
            "appointment_id",
            "patient_id"
        ).alias("a"),
        F.col("ad.appointment_id") == F.col("a.appointment_id"),
        "left"
    )
    .groupBy(
        F.col("ad.diagnosis_id").alias("diagnosis_id"),
        F.col("d.diagnosis_name").alias("diagnosis_name"),
        F.col("d.disease_category").alias("disease_category"),
        F.col("d.chronicity").alias("chronicity"),
        F.col("d.severity").alias("severity")
    )
    .agg(
        F.countDistinct(
            F.col("ad.appointment_diagnosis_id")
        ).alias("diagnosis_records"),

        F.countDistinct(
            F.col("ad.appointment_id")
        ).alias("unique_appointments"),

        F.countDistinct(
            F.col("a.patient_id")
        ).alias("unique_patients")
    )
    .orderBy(F.desc("diagnosis_records"))
)

display(diagnosis_trends)

diagnosis_id,diagnosis_name,disease_category,chronicity,severity,diagnosis_records,unique_appointments,unique_patients
1,Type 2 Diabetes Mellitus,Endocrine,Chronic,Moderate,600,500,100


### Disease Category Summary

Diagnoses are grouped by disease category to identify the overall
distribution of healthcare conditions.

In [0]:
disease_category_summary = (
    diagnosis_trends
    .groupBy("disease_category")
    .agg(
        F.sum("diagnosis_records").alias("total_diagnosis_records"),
        F.sum("unique_appointments").alias("total_appointments"),
        F.sum("unique_patients").alias("total_patients")
    )
    .orderBy(F.desc("total_diagnosis_records"))
)

display(disease_category_summary)

disease_category,total_diagnosis_records,total_appointments,total_patients
Endocrine,600,500,100


## KPI 10 — Treatment Outcomes

Treatment outcomes are analyzed using treatment status.

The analysis identifies the number and proportion of completed,
cancelled, pending, and other treatment states.

This provides insight into treatment utilization and operational
outcomes.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
treatment_outcomes = (
    spark.table("fact_treatments")
    .groupBy("treatment_status")
    .agg(
        F.countDistinct(
            "appointment_treatment_id"
        ).alias("treatment_records"),

        F.sum("quantity").alias(
            "total_quantity"
        )
    )
    .withColumn(
        "percentage_of_treatments",
        F.round(
            F.col("treatment_records")
            / F.sum("treatment_records").over(
                Window.partitionBy()
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("treatment_records"))
)

display(treatment_outcomes)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


treatment_status,treatment_records,total_quantity,percentage_of_treatments
Completed,367,723,61.17
Recommended,98,194,16.33
Started,86,156,14.33
Cancelled,49,97,8.17


## KPI 11 — Billing Trends

Billing trends are analyzed by year and month using the fact_billing
and dim_date tables.

The analysis measures billing transactions, total revenue,
insurance amount, and patient amount to identify changes in
hospital billing and revenue over time.

In [0]:
from pyspark.sql import functions as F

fact_billing = spark.table("fact_billing").alias("f")
dim_date = spark.table("dim_date").alias("dt")

billing_trends = (
    fact_billing
    .join(
        dim_date,
        F.col("f.date_key") == F.col("dt.date_key"),
        "left"
    )
    .filter(
    F.col("dt.date_key").isNotNull()
)

    .groupBy(
        F.col("dt.year").alias("year"),
        F.col("dt.month").alias("month"),
        F.col("dt.month_name").alias("month_name")
    )
    .agg(
        F.countDistinct("f.billing_id").alias("billing_transactions"),
        F.round(F.sum("f.total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("f.insurance_amount"), 2).alias("insurance_amount"),
        F.round(F.sum("f.patient_amount"), 2).alias("patient_amount")
    )
    .orderBy("year", "month")
)

display(billing_trends)

year,month,month_name,billing_transactions,total_revenue,insurance_amount,patient_amount
2024,1,January,24,1350145.06,878652.58,471492.48
2024,2,February,15,702455.27,433586.59,268868.68
2024,3,March,16,834981.57,563431.99,271549.58
2024,4,April,21,1190196.06,776326.7,413869.36
2024,5,May,15,718145.17,341779.53,376365.64
2024,6,June,17,1031644.23,578080.73,453563.5
2024,7,July,11,502938.17,340230.54,162707.63
2024,8,August,11,486290.19,238889.76,247400.43
2024,9,September,19,989866.28,706287.62,283578.66
2024,10,October,10,457283.94,260489.32,196794.62


# Project Requirement Coverage

| Requirement | Implementation |
|---|---|
| Normalized Healthcare OLTP | MySQL OLTP database with patients, doctors, departments, appointments, diagnoses, treatments, medications, laboratory, insurance, billing and payment entities |
| Automated ETL Pipeline | PySpark-based ingestion, cleansing, transformation and loading |
| OLTP to OLAP Transformation | Bronze → Silver → Gold architecture |
| Star Schema | Fact and dimension tables implemented in Databricks |
| Data Quality | Duplicate, NULL, referential-integrity, date and financial validation checks |
| Incremental / CDC Processing | Watermark-based incremental processing using updated_at |
| Patient Visit Analytics | Visit volume, completion, cancellation, waiting time and consultation duration |
| Doctor Performance | Doctor-wise visit and performance analysis |
| Department Performance | Department-wise visit and performance analysis |
| Treatment Outcomes | Treatment status and treatment utilization analysis |
| Disease Trends | Diagnosis and disease category analysis |
| Medication Usage | Prescription and medication usage analysis |
| Hospital Revenue | Revenue, billing, insurance and patient payment analysis |
| Billing Trends | Monthly billing and revenue trends |
| Insurance Claims | Claim amount, approved amount, status and approval analysis |
| Healthcare KPIs | Key operational, financial and clinical indicators |

## Architecture

MySQL OLTP → PySpark ETL → Bronze → Silver → Gold Star Schema → Healthcare Analytics

## Development Note

The project was developed using a local MySQL OLTP environment for demonstration.
The PySpark JDBC ingestion pattern can be deployed against AWS RDS MySQL
by replacing the local MySQL connection details with the AWS RDS endpoint.

## Conclusion

The implementation demonstrates an end-to-end healthcare data engineering
pipeline that transforms normalized transactional healthcare data into an
analytical Star Schema and provides healthcare, operational, financial,
billing, insurance and treatment analytics.